# Tacotron2 Training Notebook

This notebook provides a step-by-step workflow for training a Tacotron2 model for text-to-speech synthesis. Each section covers a key part of the pipeline, from data preparation to model training and evaluation.

## 1. Install and Import Dependencies
Install required packages and import all necessary libraries for Tacotron2 training.

In [ ]:
# Install dependencies (uncomment if running in Colab or a fresh environment)
# !pip install torch torchaudio matplotlib numpy librosa tqdm

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import matplotlib.pyplot as plt
from tqdm import tqdm

# If using custom modules, ensure they are in the path
import sys
sys.path.append('..')
from tacotron2 import model as taco2_model
from tacotron2 import tokenizer
from tacotron2 import dataset


## 2. Define Training Configuration
Set up all training hyperparameters, file paths, and reproducibility settings.

In [ ]:
# Set random seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Paths
DATA_ROOT = '../clartts/'  # Update as needed
METADATA_PATH = os.path.join(DATA_ROOT, 'metadata.csv')
CHECKPOINT_DIR = './checkpoints_taco2/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Training hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 100
GRAD_CLIP = 1.0

# Mel spectrogram parameters
SAMPLING_RATE = 22050
N_MELS = 80
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
FMIN = 0
FMAX = 8000


## 3. Prepare Dataset and Metadata
Load and inspect the dataset metadata, verify audio-text pairs, and preview a few samples.

In [ ]:
import pandas as pd

# Load metadata
metadata = pd.read_csv(METADATA_PATH, sep='|', header=None, names=['audio', 'transcript'])
print(f"Loaded {len(metadata)} samples.")
metadata.head()

In [ ]:
# Preview a few audio-text pairs
for idx, row in metadata.sample(3, random_state=seed).iterrows():
    print(f"Audio: {row['audio']}\nText: {row['transcript']}\n")

## 4. Clean and Encode Text
Normalize and tokenize transcripts, then convert them to integer sequences for Tacotron2 input.

In [ ]:
# Example: Use tokenizer from tacotron2/tokenizer.py
# You may need to adjust this based on your tokenizer implementation

def text_to_sequence(text):
    # Clean and tokenize text, then convert to sequence
    return tokenizer.text_to_sequence(text)

# Test encoding
sample_text = metadata.iloc[0]['transcript']
encoded = text_to_sequence(sample_text)
print(f"Original: {sample_text}\nEncoded: {encoded}")

## 5. Build Audio Preprocessing Pipeline
Load audio files, trim silence, and convert to mel spectrograms. Visualize a few examples.

In [ ]:
def load_wav(path):
    wav, sr = torchaudio.load(path)
    if sr != SAMPLING_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLING_RATE)
    return wav.squeeze(0)

def wav_to_mel(wav):
    mel_spec = torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLING_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        f_min=FMIN,
        f_max=FMAX
    )(wav)
    mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5))
    return mel_spec

# Visualize a sample
sample_audio_path = os.path.join(DATA_ROOT, 'wavs', metadata.iloc[0]['audio'])
wav = load_wav(sample_audio_path)
mel = wav_to_mel(wav)
plt.figure(figsize=(10, 4))
plt.imshow(mel.numpy(), aspect='auto', origin='lower')
plt.title('Mel Spectrogram')
plt.xlabel('Frames')
plt.ylabel('Mel bins')
plt.colorbar()
plt.show()

## 6. Create Dataset and DataLoader
Define a custom Dataset and DataLoader for efficient training.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Tacotron2Dataset(Dataset):
    def __init__(self, metadata, data_root):
        self.metadata = metadata
        self.data_root = data_root
    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        text_seq = text_to_sequence(row['transcript'])
        audio_path = os.path.join(self.data_root, 'wavs', row['audio'])
        wav = load_wav(audio_path)
        mel = wav_to_mel(wav)
        return torch.LongTensor(text_seq), mel

def collate_fn(batch):
    # Pad text and mel spectrograms
    texts, mels = zip(*batch)
    text_lens = [len(t) for t in texts]
    mel_lens = [m.shape[1] for m in mels]
    max_text = max(text_lens)
    max_mel = max(mel_lens)
    padded_texts = torch.zeros(len(batch), max_text, dtype=torch.long)
    padded_mels = torch.zeros(len(batch), N_MELS, max_mel)
    for i, (t, m) in enumerate(zip(texts, mels)):
        padded_texts[i, :len(t)] = t
        padded_mels[i, :, :m.shape[1]] = m
    return padded_texts, text_lens, padded_mels, mel_lens

# Split train/val
train_meta = metadata.sample(frac=0.95, random_state=seed)
val_meta = metadata.drop(train_meta.index)

train_dataset = Tacotron2Dataset(train_meta, DATA_ROOT)
val_dataset = Tacotron2Dataset(val_meta, DATA_ROOT)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


## 7. Initialize Tacotron2 Model
Construct the Tacotron2 model, move it to the device, and inspect parameters.

In [ ]:
# Initialize Tacotron2 model
model = taco2_model.Tacotron2()
model = model.to(device)

# Print parameter count
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tacotron2 parameters: {count_parameters(model):,}")

# Optionally load pretrained weights
# checkpoint_path = 'path/to/pretrained_checkpoint.pt'
# model.load_state_dict(torch.load(checkpoint_path, map_location=device))

## 8. Set Up Loss, Optimizer, and Scheduler
Define loss functions, optimizer, and learning rate scheduler for training.

In [ ]:
# Losses
mel_criterion = nn.MSELoss()
# If your model outputs stop tokens, add stop token loss as well
# stop_criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# For mixed precision (optional)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

## 9. Run Training Loop
Train the model, log losses, and plot progress.

In [ ]:
train_losses = []
val_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
        texts, text_lens, mels, mel_lens = [x.to(device) if torch.is_tensor(x) else x for x in batch]
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            outputs = model(texts, text_lens, mels)
            mel_out = outputs['mel_out'] if isinstance(outputs, dict) else outputs[0]
            loss = mel_criterion(mel_out, mels)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    print(f"Epoch {epoch} - Train Loss: {avg_train_loss:.4f}")
    scheduler.step()
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            texts, text_lens, mels, mel_lens = [x.to(device) if torch.is_tensor(x) else x for x in batch]
            outputs = model(texts, text_lens, mels)
            mel_out = outputs['mel_out'] if isinstance(outputs, dict) else outputs[0]
            loss = mel_criterion(mel_out, mels)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    print(f"Epoch {epoch} - Val Loss: {avg_val_loss:.4f}")
    # Plot losses
    if epoch % 5 == 0:
        plt.figure()
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.title('Training and Validation Loss')
        plt.show()
    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss
    }, os.path.join(CHECKPOINT_DIR, f'taco2_epoch{epoch}.pt'))

## 10. Validate and Save Checkpoints
Evaluate on the validation set, save model and optimizer states, and track the best checkpoint.

In [ ]:
# Track and save the best checkpoint
best_val_loss = float('inf')
best_ckpt_path = None
for epoch in range(1, EPOCHS + 1):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'taco2_epoch{epoch}.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        if ckpt['val_loss'] < best_val_loss:
            best_val_loss = ckpt['val_loss']
            best_ckpt_path = ckpt_path
print(f"Best checkpoint: {best_ckpt_path} with val loss {best_val_loss:.4f}")

## 11. Load Checkpoint and Generate Sample Mel Outputs
Reload a saved checkpoint, run inference on sample text, and visualize the generated mel spectrogram.

In [ ]:
# Load best checkpoint and run inference
if best_ckpt_path:
    model.load_state_dict(torch.load(best_ckpt_path, map_location=device)['model_state_dict'])
    model.eval()
    test_text = "This is a test sentence for Tacotron2."
    test_seq = torch.LongTensor(text_to_sequence(test_text)).unsqueeze(0).to(device)
    test_len = [test_seq.shape[1]]
    with torch.no_grad():
        outputs = model(test_seq, test_len, None)
        mel_out = outputs['mel_out'] if isinstance(outputs, dict) else outputs[0]
    mel_out = mel_out.squeeze(0).cpu().numpy()
    plt.figure(figsize=(10, 4))
    plt.imshow(mel_out, aspect='auto', origin='lower')
    plt.title('Generated Mel Spectrogram')
    plt.xlabel('Frames')
    plt.ylabel('Mel bins')
    plt.colorbar()
    plt.show()